# 07 · Constrained optimisation: a compact enclosure

**Why this is needed:** when several dimensions must satisfy a requirement, an optimiser can search for a design that improves an explicit objective while respecting constraints.

**CAS Computational Form-finding · Block Research Group · ETH Zürich**

Design a **closed rectangular box** with minimum surface area, a required internal
volume, and a maximum height. This geometric design problem connects directly to
the first notebook. COMPAS describes and displays the result; SciPy chooses its
dimensions. This is an enclosure geometry model, without wall thickness or a
structural strength calculation.

Dimensions x, y, z are in metres. Surface area is in m² and volume in m³.
**Predict:** without a height limit, which shape has the least area at a fixed volume?


## 1. Start Python
Run all cells in order with **Run → Run All Cells**, or step through with **Shift+Enter**.
JupyterLite supplies NumPy and SciPy. On desktop Jupyter, install them with
`%pip install -q numpy scipy` if an import below reports a missing package.
The pinned COMPAS packages below are the course's tested versions; `--no-deps`
avoids installing a desktop notebook server in browser Python.


In [ ]:
%pip install -q compas==2.15.1 pythreejs==2.4.2
%pip install -q compas_notebook==0.11.0 --no-deps


In [ ]:
# Enable widgets on Colab; browser JupyterLite and desktop Jupyter need no change.
try:
    from google.colab import output
except ImportError:
    pass
else:
    output.enable_custom_widget_manager()


In [ ]:
import numpy as np
from scipy.optimize import minimize
from compas.geometry import Box
from compas_notebook import Viewer
from compas_notebook.config import Config


## 2. Write the objective and the constraint
Minimise `S = 2(xy + xz + yz)` subject to `g = xyz − V = 0`.
Bounds impose positive dimensions and `z ≤ height_limit`.
The objective gradient is `[2(y+z), 2(x+z), 2(x+y)]`;
the equality-constraint gradient is `[yz, xz, xy]`.
These are analytic derivatives supplied to the optimiser with `jac`.


In [ ]:
required_volume = 3.0  # m³
height_limit = 1.0     # m
minimum_dimension = 0.1  # m

def area(dimensions):
    x, y, z = dimensions
    return 2 * (x * y + x * z + y * z)

def area_gradient(dimensions):
    x, y, z = dimensions
    return np.array([2 * (y + z), 2 * (x + z), 2 * (x + y)])

def volume_constraint(dimensions):
    return np.prod(dimensions) - required_volume

def volume_gradient(dimensions):
    x, y, z = dimensions
    return np.array([y * z, x * z, x * y])


## 3. Check the derivatives independently
A central finite difference estimates each derivative from two nearby function
values. Agreement with the analytic gradient helps catch a wrong formula before
optimisation. It is a check at a chosen point, not a mathematical proof.

Automatic differentiation could provide these gradients in a later extension;
this browser exercise uses explicit derivatives and needs no JAX installation.


In [ ]:
check_point = np.array([2.0, 1.5, 0.8])
step = 1e-6
area_fd = np.zeros(3)
volume_fd = np.zeros(3)
for i in range(3):
    delta = np.zeros(3)
    delta[i] = step
    area_fd[i] = (area(check_point + delta) - area(check_point - delta)) / (2 * step)
    volume_fd[i] = (volume_constraint(check_point + delta) - volume_constraint(check_point - delta)) / (2 * step)
assert np.allclose(area_gradient(check_point), area_fd, rtol=1e-6, atol=1e-7)
assert np.allclose(volume_gradient(check_point), volume_fd, rtol=1e-6, atol=1e-7)
print("Area gradient:", area_gradient(check_point), "finite difference:", area_fd)
print("Volume gradient:", volume_gradient(check_point), "finite difference:", volume_fd)


## 4. Run a constrained optimiser
SLSQP uses local derivative information to reduce area while satisfying the
volume equality and the dimension bounds. Start with a feasible box. Inspect
both the solver's status and the resulting constraints; success alone is not a
complete check of a design.


In [ ]:
assert required_volume > 0 and height_limit >= minimum_dimension > 0
start_height = min(height_limit, required_volume ** (1 / 3))
start = np.array([required_volume / start_height, 1.0, start_height])
bounds = [(minimum_dimension, None), (minimum_dimension, None),
          (minimum_dimension, height_limit)]
result = minimize(area, start, jac=area_gradient, method="SLSQP",
                  constraints=[{"type": "eq", "fun": volume_constraint, "jac": volume_gradient}],
                  bounds=bounds, options={"ftol": 1e-10, "maxiter": 200})
assert result.success, result.message
x, y, z = result.x
print(result.message)
print(f"Dimensions: {x:.6f} × {y:.6f} × {z:.6f} m")
print(f"Area: {result.fun:.6f} m²; volume: {np.prod(result.x):.6f} m³")
print(f"Area reduction from starting box: {area(start) - result.fun:.6f} m²")


## 5. Verify feasibility and the analytical reference
For any chosen z, the best footprint is square: `x = y = sqrt(V/z)`.
The unrestricted optimum is a cube; the height bound limits z when it is lower
than the cube side. This gives an independent reference for this particular problem.

Starting result: **x = y = √3 ≈ 1.732051 m**, **z = 1 m**,
minimum area **6 + 4√3 ≈ 12.928203 m²**. The first notebook's 2 × 1 × 1.5 m
box has the same volume and area 13 m², but exceeds this problem's height bound.


In [ ]:
assert abs(volume_constraint(result.x)) < 1e-7
assert np.all(result.x >= minimum_dimension - 1e-8)
assert z <= height_limit + 1e-8
reference_z = min(required_volume ** (1 / 3), height_limit)
reference_side = np.sqrt(required_volume / reference_z)
reference = np.array([reference_side, reference_side, reference_z])
assert np.all(reference >= minimum_dimension), "Review the reference if lower bounds become active."
assert np.allclose(result.x, reference, atol=1e-5)
assert result.fun <= area(start) + 1e-8
optimal_box = Box(float(x), float(y), float(z))
assert np.isclose(optimal_box.volume, required_volume)

config = Config()
config.view.width = 700
config.view.height = 360
config.view.show_grid = False

config.view.camera.position = [4, -6, 3]
config.view.camera.target = [0, 0, 0]
view = Viewer(config=config)
view.scene.add(optimal_box.to_mesh(), color="#5552bb", show_edges=True)
view.show()


## 6. Predict → change → check
1. Increase the height limit to 2 m. Predict whether the bound will still be active.
2. Double the required volume. Predict how the footprint changes if the 1 m height limit stays active.
3. Deliberately change one derivative; use the finite-difference check to locate the mistake, then restore it.
4. Propose a structural objective and the extra material/load assumptions it would need.

The final cell explores the analytic reference while keeping the solved design intact.


In [ ]:
trial_height_limit = 0.8  # m
trial_z = min(required_volume ** (1 / 3), trial_height_limit)
trial_side = np.sqrt(required_volume / trial_z)
trial_dimensions = np.array([trial_side, trial_side, trial_z])
assert np.isclose(np.prod(trial_dimensions), required_volume)
print("Lower enclosure reference dimensions [m]:", trial_dimensions)
print(f"Reference area: {area(trial_dimensions):.6f} m²")


## Sources
Original course optimisation example; geometric foundation:
[Seoul 2025 COMPAS](https://github.com/BRG-teaching/seoul_2025/tree/b641b732a3e337eb3a19d7a91048946820695e7a) and [COMPAS Box](https://compas.dev/compas/2.15.1/api/compas.geometry.html).
Solver and gradient contract:
[SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html)
and [SciPy minimize](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize.html).
Viewer pattern: [COMPAS JupyterLite](https://github.com/compas-dev/compas-jupyterlite/tree/d9df5aef10685c80507236554900d72b86504d6a).

Save with **Ctrl+S** (or **⌘S**), then use **File → Save and Export Notebook As → Notebook** to download your work. Browser files are local to this browser; export a copy before moving machines.
